# Research: Is Canonicalization Safe for CSP Evaluation?

This notebook tests canonicalization feasibility **without training** by treating canonicalized targets as oracle predictions and evaluating them against the original raw targets using the existing `sample_evaluation.py` pipeline.

Main question:

```text
If we replace (F_raw, L_raw) with (F_can, L_can), does sample_evaluation still match the original structure?
```

A canonicalization method is useful only if both hold:

```text
canonical_vs_raw match_rate ~= 1.0
proj_gt_can << proj_gt_raw
```


## Tests

- **Test 0:** raw self-reconstruction sanity: `(F_raw, L_raw, A)` vs itself.
- **Test 1:** canonicalized structure vs raw structure using `evaluate_csp_reconstruction`.
- **Test 2:** compare SG projection residuals `proj_gt_raw` and `proj_gt_can`.

Canonicalization methods tested:

- `niggli`
- `conventional_standard`
- `primitive_standard`
- `refined_structure`
- `refined_conventional_standard`

Methods that change atom count or species multiset are marked as not directly compatible with current KLDM CSP training.


In [1]:
from __future__ import annotations

import csv
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "kldmPlus").exists():
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.sample_evaluation.sample_evaluation import (
    aggregate_csp_reconstruction_metrics,
    build_structure_from_sample,
    evaluate_csp_reconstruction,
)
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry

try:
    from pymatgen.core import Lattice, Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
    PYMATGEN_OK = True
except Exception as exc:
    PYMATGEN_OK = False
    print("pymatgen unavailable:", type(exc).__name__, exc)

CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_diffcsp_k_x0_soft_lattice_laptop.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


{'experiment_name': 'plus_mp20_diffcsp_k_x0_soft_lattice_laptop',
 'sampler_config': '../../samplers/predictor_corrector.yaml',
 'dataset': {'task': 'csp',
  'name': 'mp20',
  'lattice_representation': 'diffcsp_k',
  'root': None,
  'batch_size': 32,
  'num_workers': 0,
  'pin_memory': False,
  'train_subset_fraction': 0.1,
  'train_subset_seed': 2002,
  'train_subset_strategy': 'balanced_space_group'},
 'time_sampler': {'type': 'uniform'},
 'model': {'eps': 1e-06,
  'lambda_l': 1.0,
  'lattice_parameterization': 'x0',
  'lattice_diffusion_type': 'VP',
  'lattice_representation': 'diffcsp_k',
  'lattice_sg_lambda': 1,
  'lattice_sg_normalize': True,
  'lattice_sg_time_weight': 'alpha_squared',
  'wrapped_normal_K': 3,
  'tdm_n_sigmas': 512,
  'tdm_compute_sigma_norm': True,
  'tdm_velocity_scale': 0.15915494309189535,
  'tdm_sigma_norm_estimator': 'quadrature',
  'tdm_sigma_norm_density_K': 20,
  'tdm_sigma_norm_grid_points': 1025,
  'tdm_sigma_norm_mc_samples': 2000,
  'score_network'

In [2]:
DEVICE = torch.device("cpu")
torch.set_grad_enabled(False)

dataset_cfg = cfg["dataset"]
model_cfg = cfg["model"]

task = CSPTask(
    dataset_name=dataset_cfg["name"],
    lattice_parameterization=model_cfg["lattice_parameterization"],
    lattice_representation=dataset_cfg["lattice_representation"],
)
root = resolve_data_root(dataset_cfg.get("root"))
lattice_transform = task.make_lattice_transform(root=root, download=False)
sym = LatticeSymmetry(eps=float(model_cfg.get("eps", 1e-8))).to(DEVICE)

print("repo", ROOT)
print("data root", root)
print("config", CONFIG_PATH)
print("lattice_representation", dataset_cfg["lattice_representation"])
print("transform", type(lattice_transform).__name__)
print("pymatgen_ok", PYMATGEN_OK)


repo /workspace
data root /workspace/data
config /workspace/configs/kldm_plus/mp_20/mp20_diffcsp_k_x0_soft_lattice_laptop.yaml
lattice_representation diffcsp_k
transform DiffCSPKContinuousIntervalLattice
pymatgen_ok True


## Helpers


In [3]:
def sg_family(sg: int) -> str:
    sg = int(sg)
    if 195 <= sg <= 230:
        return "cubic"
    if 143 <= sg <= 194:
        return "hex_trig"
    if 75 <= sg <= 142:
        return "tetragonal"
    if 16 <= sg <= 74:
        return "orthorhombic"
    if 3 <= sg <= 15:
        return "monoclinic"
    return "triclinic"


def get_structure_id(dataset, i: int) -> str | None:
    try:
        return str(dataset.data.structure_id[i])
    except Exception:
        return None


def load_raw_sg_map(split: str) -> dict[str, int]:
    path = root / "mp_20" / "raw" / f"{split}.csv"
    if not path.exists():
        return {}
    mapping = {}
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            value = row.get("spacegroup.number")
            if value not in {None, ""}:
                mapping[str(row["material_id"])] = int(float(value))
    return mapping


def k_from_cell(cell: torch.Tensor) -> torch.Tensor:
    cell = torch.as_tensor(cell, device=DEVICE, dtype=torch.float32).reshape(1, 3, 3)
    return sym.m2v(sym.de_so3(cell)).reshape(1, 6)


def proj_abs_mean(k: torch.Tensor, sg: torch.Tensor | int) -> float:
    k = torch.as_tensor(k, device=DEVICE, dtype=torch.float32).reshape(1, 6)
    sg_t = torch.as_tensor([int(torch.as_tensor(sg).reshape(-1)[0].item())], device=DEVICE, dtype=torch.long)
    proj = sym.proj_k_to_spacegroup(k, sg_t)
    return float((k - proj).abs().mean().item())


def atomic_numbers_from_structure(structure) -> torch.Tensor:
    return torch.tensor([int(site.specie.Z) for site in structure.sites], dtype=torch.long)


def sample_tensors(sample):
    return {
        "f": torch.as_tensor(sample.pos).reshape(-1, 3).float(),
        "l": torch.as_tensor(sample.l).reshape(1, 6).float(),
        "a": torch.as_tensor(sample.atomic_numbers).reshape(-1).long(),
        "sg": int(torch.as_tensor(sample.space_group).reshape(-1)[0].item()),
    }


def eval_row(result, prefix: str = "") -> dict:
    return {
        f"{prefix}valid": bool(result.valid),
        f"{prefix}match": bool(result.match),
        f"{prefix}rmse": result.rmse,
        f"{prefix}frac_rmse": result.frac_rmse,
        f"{prefix}frac_rmse_status": result.frac_rmse_status,
        f"{prefix}lengths_rmse": result.lattice_lengths_rmse,
        f"{prefix}angles_rmse": result.lattice_angles_rmse,
        f"{prefix}volume_rel_error": result.volume_rel_error,
        f"{prefix}composition_match": result.composition_match,
        f"{prefix}requested_sg_match": result.requested_space_group_match,
        f"{prefix}detected_sg": result.detected_space_group,
        f"{prefix}validity_reason": result.validity_reason,
    }


## Canonicalization Functions


In [4]:
def canonicalize_structure(structure, method: str, *, symprec: float = 0.1, angle_tolerance: float = 5.0):
    if not PYMATGEN_OK:
        raise RuntimeError("pymatgen is unavailable")
    if method == "raw":
        return structure.copy()
    if method == "niggli":
        return structure.get_reduced_structure(reduction_algo="niggli").get_sorted_structure()

    analyzer = SpacegroupAnalyzer(structure, symprec=symprec, angle_tolerance=angle_tolerance)
    if method == "conventional_standard":
        return analyzer.get_conventional_standard_structure().get_sorted_structure()
    if method == "primitive_standard":
        return analyzer.get_primitive_standard_structure().get_sorted_structure()
    if method == "refined_structure":
        return analyzer.get_refined_structure().get_sorted_structure()
    if method == "refined_conventional_standard":
        refined = analyzer.get_refined_structure()
        return SpacegroupAnalyzer(refined, symprec=symprec, angle_tolerance=angle_tolerance).get_conventional_standard_structure().get_sorted_structure()
    raise ValueError(f"Unknown canonicalization method: {method}")


CANONICALIZATION_METHODS = [
    "niggli",
    "conventional_standard",
    "primitive_standard",
    "refined_structure",
    "refined_conventional_standard",
]
CANON_SYMPREC = 0.1
CANON_ANGLE_TOLERANCE = 5.0
print(CANONICALIZATION_METHODS)


['niggli', 'conventional_standard', 'primitive_standard', 'refined_structure', 'refined_conventional_standard']


## Load Validation Samples

Start with validation because this is closest to CSP evaluation. Increase `MAX_SAMPLES` or add train/test once the notebook is working.


In [5]:
SPLIT = "val"
MAX_SAMPLES = 256

raw_sg_map = load_raw_sg_map(SPLIT)
dataset = task.fit_dataset(root=root, split=SPLIT, download=False)
num_samples = min(len(dataset), MAX_SAMPLES)

samples = []
for idx in range(num_samples):
    sample = dataset[idx]
    sid = get_structure_id(dataset, idx)
    tensors = sample_tensors(sample)
    samples.append({
        "idx": idx,
        "structure_id": sid,
        "sample": sample,
        "sg": tensors["sg"],
        "raw_csv_sg": raw_sg_map.get(sid),
        "family": sg_family(tensors["sg"]),
    })

print(f"loaded split={SPLIT} n={len(samples)}")


dataset_cache action=load dataset=mp_20 split=val reason=fresh path=/workspace/data/mp_20/processed/val
dataset_cache action=from_cache_path:start dataset=mp_20 split=val
dataset_cache action=from_cache_path:done dataset=mp_20 split=val
dataset_cache action=builder_build:start dataset=mp_20 split=val
dataset_cache action=builder_build:done dataset=mp_20 split=val
loaded split=val n=256


## Test 0: Raw Self-Reconstruction Sanity

This should be almost perfect. If this fails, stop and debug evaluation/lattice decoding before interpreting canonicalization results.


In [6]:
self_results = []
self_rows = []
for item in samples:
    sample = item["sample"]
    t = sample_tensors(sample)
    result = evaluate_csp_reconstruction(
        pred_f=t["f"],
        pred_l=t["l"],
        pred_a=t["a"],
        target_f=t["f"],
        target_l=t["l"],
        target_a=t["a"],
        lattice_transform=lattice_transform,
        requested_space_group=t["sg"],
    )
    self_results.append(result)
    self_rows.append({
        "idx": item["idx"],
        "structure_id": item["structure_id"],
        "sg": item["sg"],
        "family": item["family"],
        **eval_row(result),
    })

self_df = pd.DataFrame(self_rows)
self_metrics = aggregate_csp_reconstruction_metrics(self_results)
display(pd.Series(self_metrics, name="raw_self_reconstruction"))
display(self_df.sort_values(["match", "rmse"], ascending=[True, False]).head(20))


num_samples                              256
valid                                    1.0
match_rate                               1.0
rmse                                     0.0
rmse_defined_count                       256
frac_rmse                                0.0
frac_rmse_defined_count                  256
standardized_frac_rmse                  None
standardized_frac_rmse_defined_count       0
composition_match_rate                   1.0
requested_space_group_match_rate        None
detected_sg_agreement                   None
detected_family_agreement               None
lattice_lengths_rmse                     0.0
lattice_angles_rmse                      0.0
volume_rel_error                         0.0
matcher_diagnosis_counts                  {}
Name: raw_self_reconstruction, dtype: object

,idx,structure_id,sg,family,valid,match,rmse,frac_rmse,frac_rmse_status,lengths_rmse,angles_rmse,volume_rel_error,composition_match,requested_sg_match,detected_sg,validity_reason
140,140,mp-1282726,15,monoclinic,True,True,9.720887e-06,0.0,ok,0.0,0.0,0.0,True,None,None,ok
28,28,mp-755972,12,monoclinic,True,True,5.063450e-06,0.0,ok,0.0,0.0,0.0,True,None,None,ok
13,13,mp-1187541,139,tetragonal,True,True,2.873194e-06,0.0,ok,0.0,0.0,0.0,True,None,None,ok
18,18,mp-1224054,121,tetragonal,True,True,2.646864e-06,0.0,ok,0.0,0.0,0.0,True,None,None,ok
182,182,mp-1220534,38,orthorhombic,True,True,1.173660e-06,0.0,ok,0.0,0.0,0.0,True,None,None,ok
172,172,mp-568857,15,monoclinic,True,True,7.067497e-07,0.0,ok,0.0,0.0,0.0,True,None,None,ok
188,188,mp-12948,63,orthorhombic,True,True,5.396221e-07,0.0,ok,0.0,0.0,0.0,True,None,None,ok
126,126,mp-28029,62,orthorhombic,True,True,1.546794e-07,0.0,ok,0.0,0.0,0.0,True,None,None,ok
111,111,mp-1174026,12,monoclinic,True,True,1.101217e-07,0.0,ok,0.0,0.0,0.0,True,None,None,ok
150,150,mp-1105710,2,triclinic,True,True,1.090369e-07,0.0,ok,0.0,0.0,0.0,True,None,None,ok


## Tests 1-2: Canonicalized Structure vs Raw Target

For each canonicalization method:

1. Build raw target structure through the same evaluator lattice transform.
2. Canonicalize the raw structure.
3. Extract `F_can`, `A_can`, `L_can`, and `k_can`.
4. Evaluate `(F_can, k_can, A_can)` against `(F_raw, k_raw, A_raw)`.
5. Compare `proj_gt_raw` and `proj_gt_can`.


In [7]:
canon_results_by_method: dict[str, list] = {method: [] for method in CANONICALIZATION_METHODS}
canon_rows = []

for item in samples:
    sample = item["sample"]
    t = sample_tensors(sample)
    raw_structure = build_structure_from_sample(
        f=t["f"],
        l=t["l"],
        a=t["a"],
        lattice_transform=lattice_transform,
    )
    proj_raw = proj_abs_mean(t["l"], t["sg"])
    raw_num_atoms = int(t["a"].numel())
    raw_species_sorted = sorted(int(v) for v in t["a"].tolist())

    for method in CANONICALIZATION_METHODS:
        try:
            can_structure = canonicalize_structure(
                raw_structure,
                method,
                symprec=CANON_SYMPREC,
                angle_tolerance=CANON_ANGLE_TOLERANCE,
            )
            f_can = torch.as_tensor(can_structure.frac_coords, dtype=torch.float32)
            a_can = atomic_numbers_from_structure(can_structure)
            k_can = k_from_cell(torch.as_tensor(can_structure.lattice.matrix, dtype=torch.float32))
            can_num_atoms = int(a_can.numel())
            can_species_sorted = sorted(int(v) for v in a_can.tolist())
            atom_count_preserved = can_num_atoms == raw_num_atoms
            species_preserved = can_species_sorted == raw_species_sorted
            result = evaluate_csp_reconstruction(
                pred_f=f_can,
                pred_l=k_can,
                pred_a=a_can,
                target_f=t["f"],
                target_l=t["l"],
                target_a=t["a"],
                lattice_transform=lattice_transform,
                requested_space_group=t["sg"],
            )
            proj_can = proj_abs_mean(k_can, t["sg"])
            error = None
        except Exception as exc:
            result = None
            proj_can = None
            atom_count_preserved = False
            species_preserved = False
            can_num_atoms = None
            error = f"{type(exc).__name__}: {exc}"

        if result is not None:
            canon_results_by_method[method].append(result)
            eval_values = eval_row(result)
        else:
            eval_values = {
                "valid": False,
                "match": False,
                "rmse": None,
                "frac_rmse": None,
                "frac_rmse_status": "canonicalization_failed",
                "lengths_rmse": None,
                "angles_rmse": None,
                "volume_rel_error": None,
                "composition_match": None,
                "requested_sg_match": None,
                "detected_sg": None,
                "validity_reason": error,
            }

        canon_rows.append({
            "idx": item["idx"],
            "structure_id": item["structure_id"],
            "sg": item["sg"],
            "family": item["family"],
            "method": method,
            "raw_num_atoms": raw_num_atoms,
            "can_num_atoms": can_num_atoms,
            "atom_count_preserved": atom_count_preserved,
            "species_preserved": species_preserved,
            "directly_compatible": bool(atom_count_preserved and species_preserved),
            "proj_gt_raw": proj_raw,
            "proj_gt_can": proj_can,
            "proj_improvement": None if proj_can is None else proj_raw - proj_can,
            "proj_relative": None if proj_can is None else proj_can / max(proj_raw, 1e-12),
            "canonicalization_error": error,
            **eval_values,
        })

canon_df = pd.DataFrame(canon_rows)
display(canon_df.head())


/tmp/ipykernel_77375/1617187513.py:27: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  k_can = k_from_cell(torch.as_tensor(can_structure.lattice.matrix, dtype=torch.float32))


,idx,structure_id,sg,family,method,raw_num_atoms,can_num_atoms,atom_count_preserved,species_preserved,directly_compatible,...,rmse,frac_rmse,frac_rmse_status,lengths_rmse,angles_rmse,volume_rel_error,composition_match,requested_sg_match,detected_sg,validity_reason
0,0,mp-865981,225,cubic,niggli,4,4,True,True,True,...,3.654041e-16,0.0,ok,0.000002,0.000004,1.558273e-07,True,None,None,ok
1,0,mp-865981,225,cubic,conventional_standard,4,16,False,False,False,...,9.188863e-17,NaN,num_atoms_mismatch,2.091281,29.999986,2.999999e+00,False,None,None,ok
2,0,mp-865981,225,cubic,primitive_standard,4,4,True,True,True,...,7.330176e-17,0.0,ok,0.000003,0.000023,1.040705e-06,True,None,None,ok
3,0,mp-865981,225,cubic,refined_structure,4,16,False,False,False,...,9.188863e-17,NaN,num_atoms_mismatch,2.091282,29.999986,2.999999e+00,False,None,None,ok
4,0,mp-865981,225,cubic,refined_conventional_standard,4,16,False,False,False,...,9.188863e-17,NaN,num_atoms_mismatch,2.091281,29.999986,2.999999e+00,False,None,None,ok


## Aggregate Results by Method


In [8]:
summary_rows = []
for method in CANONICALIZATION_METHODS:
    method_df = canon_df[canon_df["method"] == method]
    results = canon_results_by_method[method]
    eval_metrics = aggregate_csp_reconstruction_metrics(results) if results else {}
    summary_rows.append({
        "method": method,
        "n": int(len(method_df)),
        "directly_compatible_rate": float(method_df["directly_compatible"].mean()),
        "atom_count_preserved_rate": float(method_df["atom_count_preserved"].mean()),
        "species_preserved_rate": float(method_df["species_preserved"].mean()),
        "mean_proj_raw": float(method_df["proj_gt_raw"].mean()),
        "mean_proj_can": float(method_df["proj_gt_can"].dropna().mean()) if method_df["proj_gt_can"].notna().any() else np.nan,
        "mean_proj_improvement": float(method_df["proj_improvement"].dropna().mean()) if method_df["proj_improvement"].notna().any() else np.nan,
        "median_proj_relative": float(method_df["proj_relative"].dropna().median()) if method_df["proj_relative"].notna().any() else np.nan,
        "valid": eval_metrics.get("valid"),
        "match_rate": eval_metrics.get("match_rate"),
        "rmse": eval_metrics.get("rmse"),
        "frac_rmse": eval_metrics.get("frac_rmse"),
        "composition_match_rate": eval_metrics.get("composition_match_rate"),
        "requested_sg_match_rate": eval_metrics.get("requested_space_group_match_rate"),
        "lengths_rmse": eval_metrics.get("lattice_lengths_rmse"),
        "angles_rmse": eval_metrics.get("lattice_angles_rmse"),
        "volume_rel_error": eval_metrics.get("volume_rel_error"),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["match_rate", "mean_proj_improvement"], ascending=[False, False])
display(summary_df)


,method,n,directly_compatible_rate,atom_count_preserved_rate,species_preserved_rate,mean_proj_raw,mean_proj_can,mean_proj_improvement,median_proj_relative,valid,match_rate,rmse,frac_rmse,composition_match_rate,requested_sg_match_rate,lengths_rmse,angles_rmse,volume_rel_error
0,niggli,256,1.000000,1.000000,1.000000,0.079406,0.079406,-5.319393e-08,1.000000e+00,1.000000,1.000000,2.447626e-07,0.037399,1.000000,None,0.000004,0.002462,5.305937e-07
2,primitive_standard,256,0.996094,0.996094,0.996094,0.079406,0.100088,-2.068216e-02,1.000000e+00,1.000000,0.996094,7.580640e-05,0.106936,0.996094,None,0.884816,11.910353,3.349402e-03
3,refined_structure,256,0.425781,0.425781,0.425781,0.079406,0.000253,7.915311e-02,5.388672e-08,0.996094,0.988281,7.640083e-05,0.104153,0.425781,None,2.432641,15.123904,1.010602e+00
1,conventional_standard,256,0.425781,0.425781,0.425781,0.079406,0.000253,7.915309e-02,2.028885e-07,0.996094,0.988281,7.640152e-05,0.102912,0.425781,None,2.188611,15.533442,1.010602e+00
4,refined_conventional_standard,256,0.425781,0.425781,0.425781,0.079406,0.000506,7.890022e-02,2.028885e-07,0.996094,0.988281,7.640170e-05,0.103249,0.425781,None,2.188682,15.523244,1.010067e+00


## Breakdown by Crystal Family


In [9]:
family_summary = canon_df.groupby(["method", "family"]).agg(
    n=("idx", "size"),
    directly_compatible_rate=("directly_compatible", "mean"),
    valid=("valid", "mean"),
    match_rate=("match", "mean"),
    mean_rmse=("rmse", "mean"),
    mean_frac_rmse=("frac_rmse", "mean"),
    mean_proj_raw=("proj_gt_raw", "mean"),
    mean_proj_can=("proj_gt_can", "mean"),
    mean_proj_improvement=("proj_improvement", "mean"),
    median_proj_relative=("proj_relative", "median"),
    mean_abs_volume_rel_error=("volume_rel_error", lambda s: s.dropna().abs().mean() if s.dropna().shape[0] else np.nan),
).reset_index().sort_values(["method", "mean_proj_raw"], ascending=[True, False])
display(family_summary)


,method,family,n,directly_compatible_rate,valid,match_rate,mean_rmse,mean_frac_rmse,mean_proj_raw,mean_proj_can,mean_proj_improvement,median_proj_relative,mean_abs_volume_rel_error
0,conventional_standard,cubic,63,0.190476,1.000000,1.000000,5.787729e-08,0.019273,0.132257,3.941083e-08,1.322571e-01,3.078862e-07,2.301587e+00
1,conventional_standard,hex_trig,43,0.674419,0.976744,0.976744,1.369463e-05,0.094215,0.108435,1.439349e-08,1.084354e-01,9.872301e-08,6.511691e-01
4,conventional_standard,tetragonal,37,0.405405,1.000000,0.972973,1.900750e-05,0.088683,0.077688,4.573030e-08,7.768766e-02,5.088589e-07,5.945943e-01
2,conventional_standard,monoclinic,39,0.384615,1.000000,0.974359,2.973980e-04,0.160659,0.045237,2.862766e-08,4.523716e-02,3.970254e-07,6.080482e-01
3,conventional_standard,orthorhombic,64,0.437500,1.000000,1.000000,1.028787e-04,0.136901,0.042099,1.011516e-03,4.108720e-02,0.000000e+00,6.250012e-01
5,conventional_standard,triclinic,10,1.000000,1.000000,1.000000,1.811284e-05,0.068055,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,4.954777e-07
6,niggli,cubic,63,1.000000,1.000000,1.000000,7.244463e-08,0.036507,0.132257,1.322571e-01,-1.769858e-08,1.000000e+00,4.243530e-07
7,niggli,hex_trig,43,1.000000,1.000000,1.000000,3.543532e-07,0.063342,0.108435,1.084354e-01,1.996616e-08,1.000000e+00,6.001554e-07
10,niggli,tetragonal,37,1.000000,1.000000,1.000000,3.197124e-07,0.025668,0.077688,7.768777e-02,-6.483410e-08,9.999998e-01,5.710452e-07
8,niggli,monoclinic,39,1.000000,1.000000,1.000000,4.632125e-07,0.014527,0.045237,4.523738e-02,-1.872969e-07,1.000000e+00,6.914737e-07


## Inspect Failures and Best Improvements


In [10]:
display(canon_df[~canon_df["match"]].sort_values(["method", "proj_improvement"], ascending=[True, False])[[
    "method", "idx", "structure_id", "sg", "family",
    "directly_compatible", "raw_num_atoms", "can_num_atoms", "species_preserved",
    "valid", "match", "rmse", "frac_rmse", "frac_rmse_status",
    "proj_gt_raw", "proj_gt_can", "proj_improvement", "proj_relative",
    "validity_reason", "canonicalization_error",
]].head(80))

best_safe = canon_df[(canon_df["match"]) & (canon_df["directly_compatible"])].copy()
display(best_safe.sort_values("proj_improvement", ascending=False)[[
    "method", "idx", "structure_id", "sg", "family",
    "rmse", "frac_rmse", "proj_gt_raw", "proj_gt_can", "proj_improvement", "proj_relative",
    "lengths_rmse", "angles_rmse", "volume_rel_error",
]].head(80))


,method,idx,structure_id,sg,family,directly_compatible,raw_num_atoms,can_num_atoms,species_preserved,valid,match,rmse,frac_rmse,frac_rmse_status,proj_gt_raw,proj_gt_can,proj_improvement,proj_relative,validity_reason,canonicalization_error
1126,conventional_standard,225,mp-4651,140,tetragonal,False,10,20,False,True,False,NaN,NaN,num_atoms_mismatch,0.166730,3.853429e-08,0.166730,2.311184e-07,ok,None
1081,conventional_standard,216,mp-627401,160,hex_trig,False,15,45,False,False,False,NaN,NaN,num_atoms_mismatch,0.135843,2.877546e-09,0.135843,2.118286e-08,huge_lattice,None
806,conventional_standard,161,mp-1104341,12,monoclinic,False,7,2,False,True,False,NaN,NaN,num_atoms_mismatch,0.023551,9.378025e-09,0.023551,3.982050e-07,ok,None
807,primitive_standard,161,mp-1104341,12,monoclinic,False,7,1,False,True,False,NaN,NaN,num_atoms_mismatch,0.023551,1.019822e-01,-0.078431,4.330318e+00,ok,None
1129,refined_conventional_standard,225,mp-4651,140,tetragonal,False,10,20,False,True,False,NaN,NaN,num_atoms_mismatch,0.166730,3.853429e-08,0.166730,2.311184e-07,ok,None
1084,refined_conventional_standard,216,mp-627401,160,hex_trig,False,15,45,False,False,False,NaN,NaN,num_atoms_mismatch,0.135843,2.877546e-09,0.135843,2.118286e-08,huge_lattice,None
809,refined_conventional_standard,161,mp-1104341,12,monoclinic,False,7,3,False,True,False,NaN,NaN,num_atoms_mismatch,0.023551,6.473634e-02,-0.041186,2.748802e+00,ok,None
1128,refined_structure,225,mp-4651,140,tetragonal,False,10,20,False,True,False,NaN,NaN,num_atoms_mismatch,0.166730,8.466184e-09,0.166730,5.077793e-08,ok,None
1083,refined_structure,216,mp-627401,160,hex_trig,False,15,45,False,False,False,NaN,NaN,num_atoms_mismatch,0.135843,2.877546e-09,0.135843,2.118286e-08,huge_lattice,None
808,refined_structure,161,mp-1104341,12,monoclinic,False,7,2,False,True,False,NaN,NaN,num_atoms_mismatch,0.023551,9.378025e-09,0.023551,3.982050e-07,ok,None


,method,idx,structure_id,sg,family,rmse,frac_rmse,proj_gt_raw,proj_gt_can,proj_improvement,proj_relative,lengths_rmse,angles_rmse,volume_rel_error
279,refined_conventional_standard,55,mp-1078813,189,hex_trig,2.245318e-16,0.237220,0.197903,1.947569e-08,0.197903,9.841029e-08,2.880613,24.494895,4.559167e-07
278,refined_structure,55,mp-1078813,189,hex_trig,2.245318e-16,0.237220,0.197903,1.947569e-08,0.197903,9.841029e-08,2.880613,24.494895,4.559167e-07
276,conventional_standard,55,mp-1078813,189,hex_trig,2.245318e-16,0.237220,0.197903,1.947569e-08,0.197903,9.841029e-08,2.880613,24.494895,4.559167e-07
277,primitive_standard,55,mp-1078813,189,hex_trig,2.245318e-16,0.237220,0.197903,1.947569e-08,0.197903,9.841029e-08,2.880613,24.494895,4.559167e-07
24,refined_conventional_standard,4,mp-20332,189,hex_trig,1.252119e-08,0.240231,0.192815,1.741122e-08,0.192814,9.030036e-08,2.806986,24.494884,2.088593e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1121,conventional_standard,224,mp-625415,11,monoclinic,1.013759e-07,0.200912,0.061356,0.000000e+00,0.061356,0.000000e+00,2.300018,23.325673,8.201424e-09
481,conventional_standard,96,mp-1873,136,tetragonal,1.636457e-08,0.140187,0.048399,5.729932e-09,0.048399,1.183893e-07,1.320678,0.000000,8.707234e-07
484,refined_conventional_standard,96,mp-1873,136,tetragonal,1.636457e-08,0.140187,0.048399,5.729932e-09,0.048399,1.183893e-07,1.320678,0.000000,8.707234e-07
482,primitive_standard,96,mp-1873,136,tetragonal,1.636457e-08,0.140187,0.048399,5.729932e-09,0.048399,1.183893e-07,1.320678,0.000000,8.707234e-07


## Decision Checklist

Canonicalization is feasible for current KLDM CSP only if both are true:

```text
canonicalized_vs_raw match_rate ~= 1.0
proj_gt_can << proj_gt_raw
```

Interpretation:

- `match_rate=1`, `rmse ~= 0`, `proj_gt_can << proj_gt_raw`: best case. Safe and useful.
- `match_rate=1`, `rmse ~= 0`, `proj_gt_can ~= proj_gt_raw`: safe but not useful for the DiffCSP++ mask.
- `match_rate<1`, `proj_gt_can << proj_gt_raw`: dangerous. It improves mask compatibility by changing the CSP target.
- `num_atoms` changes: not directly compatible with current batching/composition assumptions.
- high `frac_rmse` with `match_rate=1` and `rmse ~= 0`: usually okay; the fractional chart changed but physical structure matched.
